# Importations nécessaires

In [10]:
import numpy as np
from gurobipy import *
import pandas as pd
from collections import defaultdict

# Préparation des données
Récuperation des matrices de gains, de coût, des listes des bâtiments, de leurs types et des durées de travaux.

In [11]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("dataset_efficacity_avec_duree.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "typo_dpe", "id_simulation", "conso_total_mwh_an"]]
)

# Comptage par catégorie
nb_buildings_by_type = (
    df_calibrated
    .groupby("typo_dpe")["building_name"]
    .count()
    .to_dict()
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================
df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# --- 2. Groupement ---
group_gains    = df_nc.groupby("building_name")["gains_totaux_mwh_an"].apply(list)
group_cost     = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)
group_duration = df_nc.groupby("building_name")["temps_de_travaux"].apply(list)  # en mois

# --- 3. Taille maximale ---
max_len = max(
    group_gains.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
gains_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_gains
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

group_type = (
    df_nc
    .groupby("building_name")["typo_dpe"]
    .first()
)

# --- 5. Métadonnées utiles ---
building_names = group_gains.index.tolist()
building_types = group_type.loc[building_names].tolist()

grouped = df_nc.groupby("building_name", sort=False)["id_simulation"].apply(list)
renovation_names_per_building = grouped.to_dict()

# Durée par simulation (en mois)
duration_by_sim = {
    row["id_simulation"]: int(row["temps_de_travaux"])
    for _, row in df_nc.iterrows()
}

df_calibrated = df_calibrated[
    df_calibrated["building_name"].isin(building_names)
]

# Isolation des travaux spécifiques (utile pour l'excel)
renov_cols = df.columns[6:14]
works_by_sim = {}
for _, row in df_nc.iterrows():
    works_by_sim[row["id_simulation"]] = {
        col: bool(row[col]) for col in renov_cols
    }

## Construction du graphe de transitions

In [12]:
# ============================
# PARTIE C — TRANSITION GRAPH
# ============================

transition_graph = {}

for building, group in df_nc.groupby("building_name", sort=False):

    group = group.reset_index(drop=True)

    works = group[renov_cols].astype(bool).values
    costs = group["cout_investissement_euros"].values
    gains = group["gains_totaux_mwh_an"].values
    ids   = group["id_simulation"].tolist()
    durs  = group["temps_de_travaux"].astype(int).values

    n = len(group)

    compat = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if np.all(~works[j] | works[i]):
                compat[i, j] = 1

    transitions = []

    # état initial
    for i in range(n):
        transitions.append({
            "from":     "none",
            "to":       ids[i],
            "cost":     costs[i],
            "gain":     gains[i],
            "duration": int(durs[i])
        })

    # transitions entre rénovations (j ⊂ i)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if compat[i, j] == 1 and compat[j, i] == 0:
                transitions.append({
                    "from":     ids[j],
                    "to":       ids[i],
                    "cost":     costs[i] - costs[j],
                    "gain":     gains[i] - gains[j],
                    "duration": int(durs[i])
                })

    transition_graph[building] = transitions

edges = {b: list(range(len(transition_graph[b]))) for b in building_names}

cost_e       = {}
gain_e       = {}
from_state   = {}
to_state     = {}
duration_e   = {}

for b in building_names:
    for e, edge in enumerate(transition_graph[b]):
        cost_e[b, e]     = edge["cost"]
        gain_e[b, e]     = edge["gain"]
        from_state[b, e] = edge["from"]
        to_state[b, e]   = edge["to"]
        duration_e[b, e] = edge["duration"]

## Construction du graphe temporel

Chaque nœud est un couple **(état énergétique, mois)**.

Deux types d'arcs :
- **Attente** : `(r, t) → (r, t+1)` — le bâtiment reste dans son état, coût/gain nuls.
- **Rénovation** : `(r, t) → (r', t+d_e)` — le bâtiment passe de l'état `r` à `r'` en `d_e` mois.

La conservation de flux à chaque nœud garantit nativement qu'on ne quitte un état qu'après y être **réellement arrivé** (le saut temporel `t+d_e` rend les voyages dans le futur impossibles).

In [13]:
T          = 300  # horizon en mois (jan 2026 → déc 2050)
nbr_annees = 25
years      = list(range(2026, 2051))

# Conso totale avec sobriété 10%
Conso_total = 0.9 * np.sum(df_calibrated["conso_total_mwh_an"])
alpha       = 0.5  # fraction du coût payée au démarrage

print(f"Conso totale (sobriété 10%) : {Conso_total:.1f} MWh/an")
print(f"Horizon : {T} mois ({nbr_annees} ans)")

temporal_arcs = {b: [] for b in building_names}

for b in building_names:

    states = set(["none"])
    for e in edges[b]:
        states.add(from_state[b, e])
        states.add(to_state[b, e])

    # --- arcs attente ---
    for r in states:
        for t in range(T):
            temporal_arcs[b].append({
                "type": "wait",
                "from": (r, t),
                "to":   (r, t + 1),
                "cost": 0,
                "gain": 0,
                "edge": None
            })

    # --- arcs rénovation ---
    for e in edges[b]:
        r_from = from_state[b, e]
        r_to   = to_state[b, e]
        d      = duration_e[b, e]
        for t in range(T):
            if t + d <= T:
                temporal_arcs[b].append({
                    "type": "renov",
                    "edge": e,
                    "from": (r_from, t),
                    "to":   (r_to, t + d),
                    "cost": cost_e[b, e],
                    "gain": gain_e[b, e]
                })

total_arcs = sum(len(temporal_arcs[b]) for b in building_names)
print(f"Nombre total d'arcs : {total_arcs}")

Conso totale (sobriété 10%) : 18759.1 MWh/an
Horizon : 300 mois (25 ans)
Nombre total d'arcs : 1001960


In [14]:
# Index d'accès rapide : nœud → liste d'arcs entrants / sortants
incoming = {b: {} for b in building_names}
outgoing = {b: {} for b in building_names}

for b in building_names:
    for arc_id, arc in enumerate(temporal_arcs[b]):
        outgoing[b].setdefault(arc["from"], []).append(arc_id)
        incoming[b].setdefault(arc["to"],   []).append(arc_id)

## Modèle Gurobi

Variable : $f_{b,a} \in \{0,1\}$ — le flux du bâtiment $b$ emprunte l'arc $a$.

In [15]:
m = Model("Graphe temporel - Transition énergétique")
f = {}

for b in building_names:
    for a in range(len(temporal_arcs[b])):
        f[b, a] = m.addVar(vtype=GRB.BINARY, name=f"f_{b}_{a}")

m.update()
print(f"Variables binaires : {m.NumVars}")

Set parameter Username
Set parameter LicenseID to value 2773884
Academic license - for non-commercial use only - expires 2027-02-02
Variables binaires : 1001960


### Contrainte de conservation de flux

Pour tout nœud $(r, t)$ de chaque bâtiment $b$ :
$$\sum_{a \in \delta^-(r,t)} f_{b,a} = \sum_{a \in \delta^+(r,t)} f_{b,a}$$

**Sauf** pour le nœud source $(\text{None}, 0)$ et les nœuds puits $(r, T)$.

In [16]:
for b in building_names:
    all_nodes = set(list(incoming[b].keys()) + list(outgoing[b].keys()))

    for node in all_nodes:
        r, t = node
        # Nœud source et nœuds puits : exclus de la conservation
        if node == ("none", 0) or t == T:
            continue

        m.addConstr(
            quicksum(f[b, a] for a in incoming[b].get(node, []))
            ==
            quicksum(f[b, a] for a in outgoing[b].get(node, [])),
            name=f"flow_{b}_{r}_{t}"
        )

### Contrainte source : chaque bâtiment démarre depuis (none, 0)

$$\sum_{a \in \delta^+((\text{none},\, 0))} f_{b,a} = 1$$

In [17]:
for b in building_names:
    source_node = ("none", 0)
    out_source  = outgoing[b].get(source_node, [])

    m.addConstr(
        quicksum(f[b, a] for a in out_source) == 1,
        name=f"source_{b}"
    )

### Contrainte puits : le flux arrive bien en T

Pour chaque bâtiment, la somme des flux entrant dans les nœuds $(r, T)$ vaut 1 (un seul état final).

$$\sum_{r} \sum_{a \in \delta^-((r,\,T))} f_{b,a} = 1$$

In [18]:
for b in building_names:
    sink_arcs = [
        a for a, arc in enumerate(temporal_arcs[b])
        if arc["to"][1] == T
    ]
    if sink_arcs:
        m.addConstr(
            quicksum(f[b, a] for a in sink_arcs) == 1,
            name=f"sink_{b}"
        )

### Contrainte de budget annuel

Une fraction $\alpha$ du coût est payée au **démarrage** (mois $t$) et $1-\alpha$ à la **fin** (mois $t + d_e$) :

$$\forall y \quad \sum_b \sum_{a \in A_y^{\text{start}}} \alpha \cdot c_a \cdot f_{b,a}
+ \sum_b \sum_{a \in A_y^{\text{end}}} (1-\alpha) \cdot c_a \cdot f_{b,a} \leq 2\,\text{M€}$$

où $A_y^{\text{start}}$ (resp. $A_y^{\text{end}}$) sont les arcs de rénovation dont le mois de **démarrage** (resp. de **fin**) appartient à l'année $y$.

In [19]:
budget_annuel = 2e6

def month_to_year_idx(t):
    """Retourne l'index d'année (0 = 2026) du mois t."""
    return t // 12

for a_idx in range(nbr_annees):
    months_a = set(range(a_idx * 12, (a_idx + 1) * 12))

    # Arcs de rénovation dont le démarrage est dans l'année a_idx
    start_terms = [
        alpha * temporal_arcs[b][a]["cost"] * f[b, a]
        for b in building_names
        for a, arc in enumerate(temporal_arcs[b])
        if arc["type"] == "renov" and arc["from"][1] in months_a
    ]

    # Arcs de rénovation dont la fin est dans l'année a_idx
    end_terms = [
        (1 - alpha) * temporal_arcs[b][a]["cost"] * f[b, a]
        for b in building_names
        for a, arc in enumerate(temporal_arcs[b])
        if arc["type"] == "renov" and arc["to"][1] in months_a
    ]

    all_terms = start_terms + end_terms
    if all_terms:
        m.addConstr(
            quicksum(all_terms) <= budget_annuel,
            name=f"budget_{a_idx}"
        )

### Contrainte de disponibilité par catégorie

Au mois $t$, au plus 20 % des bâtiments d'une catégorie peuvent être **en rénovation** (i.e. leur arc actif est un arc rénovation dont le démarrage $\tau$ vérifie $\tau \leq t < \tau + d_e$) :

$$\forall \text{cat}, \forall t \quad
\sum_{b \in \text{cat}} \sum_{\substack{a : \text{arc rénovation} \\ \text{from}[1] \leq t < \text{to}[1]}} f_{b,a}
\leq 0.2 \cdot |\text{cat}|$$

**Exception :** en janvier, juillet et août, la contrainte ne s'applique **pas** aux bâtiments de type **Enseignement**.

In [20]:
buildings_by_cat = defaultdict(list)
building_type_map = dict(zip(building_names, building_types))

for b_name, cat in zip(building_names, building_types):
    buildings_by_cat[cat].append(b_name)

EXEMPT_MONTHS_OF_YEAR = {0, 6, 7}   # janvier=0, juillet=6, août=7
EXEMPT_CATEGORY       = "Enseignement"

# Pré-calcul : pour chaque (b, arc de rénovation), l'ensemble des mois couverts
# arc couvre les mois [t_start, t_end - 1]
# On indexe : pour chaque mois t, quels arcs de rénovation le couvrent ?
# On construit le dictionnaire renov_arcs_covering[b][t] = liste d'arc_ids

renov_arcs_covering = {b: defaultdict(list) for b in building_names}

for b in building_names:
    for a, arc in enumerate(temporal_arcs[b]):
        if arc["type"] == "renov":
            t_start = arc["from"][1]
            t_end   = arc["to"][1]     # mois de fin (exclu)
            for t in range(t_start, t_end):
                renov_arcs_covering[b][t].append(a)

for cat, b_names in buildings_by_cat.items():
    if cat == "Autres":
        continue

    card_cat = len(b_names)

    for t in range(T):
        month_in_year = t % 12

        # Exception Enseignement en jan/jul/août
        if cat == EXEMPT_CATEGORY and month_in_year in EXEMPT_MONTHS_OF_YEAR:
            continue

        terms = [
            f[b, a]
            for b in b_names
            for a in renov_arcs_covering[b].get(t, [])
        ]

        if terms:
            m.addConstr(
                quicksum(terms) <= 0.2 * card_cat,
                name=f"dispo_{cat}_{t}"
            )

### Fonction objectif

Minimiser les écarts pondérés aux jalons du décret tertiaire.
Le gain d'un arc rénovation est pris en compte au **mois de fin** `to[1]` :

$$\min \; 0.25\bigl(0.4\,C_0 - G_{2030}\bigr)
+ 0.25\bigl(0.5\,C_0 - G_{2040}\bigr)
+ 0.50\bigl(0.6\,C_0 - G_{2050}\bigr)$$

où $G_{y} = \sum_{b}\sum_{a : \text{renov, fin} \leq T_y} \text{gain}_a \cdot f_{b,a}$.

In [21]:
# Dernier mois inclus dans chaque jalon
T_2030 = (2030 - 2026 + 1) * 12 - 1   # = 59
T_2040 = (2040 - 2026 + 1) * 12 - 1   # = 179
T_2050 = T                             # = 300  (fin = mois 300 inclus)

def gain_expr(T_jalon):
    """Expression linéaire des gains dont la fin de rénovation est <= T_jalon."""
    return quicksum(
        arc["gain"] * f[b, a]
        for b in building_names
        for a, arc in enumerate(temporal_arcs[b])
        if arc["type"] == "renov" and arc["to"][1] <= T_jalon
    )

m.setObjective(
    0.25 * (0.4 * Conso_total - gain_expr(T_2030))
    + 0.25 * (0.5 * Conso_total - gain_expr(T_2040))
    + 0.50 * (0.6 * Conso_total - gain_expr(T_2050)),
    GRB.MINIMIZE
)

In [32]:
print("Nombre de variables du modèle: ", m.NumVars)
print("Nombre de contraintes du modèle: ", m.NumConstrs)

Nombre de variables du modèle:  1001960
Nombre de contraintes du modèle:  257721


### Résolution

In [22]:
m.setParam("TimeLimit", 2 * 60)
m.update()
m.optimize()

Set parameter TimeLimit to value 120
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120

Optimize a model with 257721 rows, 1001960 columns and 4577762 nonzeros (Min)
Model fingerprint: 0x67c4cf53
Model has 743976 linear objective coefficients and an objective constant of 9.8485430878689222e+03
Variable types: 0 continuous, 1001960 integer (1001960 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+06]
  Objective range  [5e-06, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]

Presolve removed 80146 rows and 84713 columns (presolve time = 5s)...
Presolve removed 80546 rows and 130014 columns (presolve time = 10s)...
Presolve removed 80546 rows and 130014 columns (presolve time = 15s)...
Presolve removed 80

### Objectifs réalisés

In [23]:
print(f"Valeur objectif : {m.ObjVal:.4f}")

def realized_gain(T_jalon):
    return sum(
        arc["gain"] * f[b, a].X
        for b in building_names
        for a, arc in enumerate(temporal_arcs[b])
        if arc["type"] == "renov"
        and arc["to"][1] <= T_jalon
        and f[b, a].X > 0.5
    )

obj_2030 = 0.1 + 0.9 * realized_gain(T_2030) / Conso_total
obj_2040 = 0.1 + 0.9 * realized_gain(T_2040) / Conso_total
obj_2050 = 0.1 + 0.9 * realized_gain(T_2050) / Conso_total

print(f"Réduction 2030 : {obj_2030*100:.1f}%  (cible 40%)")
print(f"Réduction 2040 : {obj_2040*100:.1f}%  (cible 50%)")
print(f"Réduction 2050 : {obj_2050*100:.1f}%  (cible 60%)")

Valeur objectif : 1567.2487
Réduction 2030 : 37.0%  (cible 40%)
Réduction 2040 : 51.7%  (cible 50%)
Réduction 2050 : 55.1%  (cible 60%)


---
## Export Excel : plan_renovation_solution.xlsx

⚠️ Ne pas ré-exécuter sans nécessité.

In [24]:
def works_added(from_s, to_s):
    """Travaux réalisés par la transition from_s → to_s."""
    if from_s == "none":
        return [col for col, v in works_by_sim[to_s].items() if v]
    return [
        col for col in renov_cols
        if works_by_sim[to_s][col] and not works_by_sim[from_s][col]
    ]

In [25]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

# ============================================================
# 1. COLLECTE DES RÉSULTATS
# ============================================================
rows = []
for b in building_names:
    for a, arc in enumerate(temporal_arcs[b]):
        if arc["type"] != "renov":
            continue
        if f[b, a].X < 0.5:
            continue

        t_start = arc["from"][1]
        t_end   = arc["to"][1]
        r_from  = arc["from"][0]
        r_to    = arc["to"][0]

        year_start  = 2026 + t_start // 12
        month_start = (t_start % 12) + 1
        year_end    = 2026 + t_end // 12
        month_end   = (t_end % 12) + 1
        duree_mois  = t_end - t_start

        added = works_added(r_from, r_to)

        rows.append({
            "batiment":     b,
            "type":         building_type_map[b],
            "annee_debut":  year_start,
            "mois_debut":   month_start,
            "annee_fin":    year_end,
            "mois_fin":     month_end,
            "duree_mois":   duree_mois,
            "etat_avant":   r_from,
            "etat_apres":   r_to,
            "travaux":      ", ".join(str(w) for w in added),
            "gain_mwh_an":  round(arc["gain"], 2),
            "cout_euros":   round(arc["cost"], 0),
        })

df_sol = pd.DataFrame(rows).sort_values(["batiment", "annee_debut", "mois_debut"])

# ============================================================
# 2. STATS ANNUELLES
# ============================================================
stats_rows = []
for a_idx, yr in enumerate(years):
    yr_start = df_sol[df_sol["annee_debut"] == yr]
    yr_end   = df_sol[df_sol["annee_fin"]   == yr]

    gain_yr    = yr_end["gain_mwh_an"].sum()
    cost_start = alpha * yr_start["cout_euros"].sum()
    cost_end   = (1 - alpha) * yr_end["cout_euros"].sum()

    stats_rows.append({
        "annee":              yr,
        "chantiers_demarres": len(yr_start),
        "gain_realise_mwh":   round(gain_yr, 1),
        "cout_total_euros":   round(cost_start + cost_end, 0),
        "budget_disponible":  2_000_000,
        "budget_restant":     round(2_000_000 - cost_start - cost_end, 0),
    })

df_stats = pd.DataFrame(stats_rows)
df_stats["gain_cumule_mwh"] = df_stats["gain_realise_mwh"].cumsum()
df_stats["reduction_pct"]   = (0.1 + 0.9 * df_stats["gain_cumule_mwh"] / Conso_total).round(4)
cibles = {2030: 0.40, 2040: 0.50, 2050: 0.60}
df_stats["cible_decret"] = df_stats["annee"].map(lambda y: cibles.get(y, ""))

# ============================================================
# 3. GANTT ANNUEL
# ============================================================
gantt_data = {b: {yr: "" for yr in years} for b in building_names}
for _, row in df_sol.iterrows():
    b = row["batiment"]
    for yr in years:
        if row["annee_debut"] <= yr <= row["annee_fin"]:
            gantt_data[b][yr] = "EN TRAVAUX"

df_gantt = pd.DataFrame(gantt_data).T
df_gantt.index.name = "batiment"
df_gantt.insert(0, "type", [building_type_map[b] for b in df_gantt.index])

# ============================================================
# 4. DISPONIBILITÉ PAR CATÉGORIE
# ============================================================
avail_rows = []
for cat in buildings_by_cat:
    card = nb_buildings_by_type.get(cat, 1)
    for yr in years:
        reno = df_sol[
            (df_sol["type"] == cat) &
            (df_sol["annee_debut"] <= yr) &
            (df_sol["annee_fin"]   >= yr)
        ]["batiment"].nunique()
        avail_rows.append({
            "categorie":        cat,
            "annee":            yr,
            "nb_batiments":     card,
            "en_renovation":    reno,
            "taux_utilisation": round(reno / card, 4) if card else 0,
            "seuil_max_20pct":  round(0.2 * card, 2),
        })
df_avail = pd.DataFrame(avail_rows)

# ============================================================
# 5. RÉSUMÉ OBJECTIFS
# ============================================================
df_summary = pd.DataFrame({
    "Jalon":       ["2030", "2040", "2050"],
    "Cible":       ["40%",  "50%",  "60%"],
    "Réalisé":     [f"{obj_2030*100:.1f}%", f"{obj_2040*100:.1f}%", f"{obj_2050*100:.1f}%"],
    "Écart (pp)":  [round((obj_2030-0.40)*100,1), round((obj_2040-0.50)*100,1), round((obj_2050-0.60)*100,1)],
    "Statut":      [
        "✓ Atteint" if obj_2030 >= 0.40 else "✗ Non atteint",
        "✓ Atteint" if obj_2040 >= 0.50 else "✗ Non atteint",
        "✓ Atteint" if obj_2050 >= 0.60 else "✗ Non atteint",
    ]
})

# ============================================================
# 6. ÉCRITURE EXCEL
# ============================================================
OUTPUT_FILE = "plan_renovation_solution.xlsx"

C_HEADER_BG  = "1F3864"
C_HEADER_FG  = "FFFFFF"
C_STRIPE1    = "EBF0FA"
C_STRIPE2    = "FFFFFF"
C_RENO       = "FFC000"
C_FREE       = "E2EFDA"
C_TARGET_OK  = "C6EFCE"
C_TARGET_NOK = "FFCCCC"

thin   = Side(style="thin",   color="BBBBBB")
thick  = Side(style="medium", color="888888")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(ws):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=C_HEADER_BG)
        cell.font      = Font(bold=True, color=C_HEADER_FG, size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border
    ws.row_dimensions[1].height = 28

def auto_width(ws, min_w=8, max_w=40):
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(w+2, min_w), max_w)

def stripe_rows(ws, start=2, n_rows=None):
    for i, row in enumerate(ws.iter_rows(min_row=start)):
        if n_rows and i >= n_rows:
            break
        fill = PatternFill("solid", fgColor=C_STRIPE1 if i%2==0 else C_STRIPE2)
        for cell in row:
            if cell.fill.fgColor.rgb in ("00000000","FFFFFFFF","00FFFFFF"):
                cell.fill = fill
            cell.border = border

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    # Feuille 0 : Résumé objectifs
    df_summary.to_excel(writer, sheet_name="0_Résumé_Objectifs", index=False)
    ws0 = writer.sheets["0_Résumé_Objectifs"]
    style_header(ws0)
    auto_width(ws0)
    for i, row_ws in enumerate(ws0.iter_rows(min_row=2, max_row=4)):
        ok = df_summary.iloc[i]["Statut"].startswith("✓")
        for cell in row_ws:
            cell.fill      = PatternFill("solid", fgColor=C_TARGET_OK if ok else C_TARGET_NOK)
            cell.font      = Font(bold=True, size=12)
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border    = border
        ws0.row_dimensions[i+2].height = 22

    # Feuille 1 : Détail transitions
    df_sol.to_excel(writer, sheet_name="1_Transitions", index=False)
    ws1 = writer.sheets["1_Transitions"]
    style_header(ws1)
    stripe_rows(ws1, start=2, n_rows=len(df_sol))
    auto_width(ws1)
    ws1.freeze_panes = "A2"

    # Feuille 2 : Gantt annuel
    df_gantt.to_excel(writer, sheet_name="2_Gantt_Annuel")
    ws2 = writer.sheets["2_Gantt_Annuel"]
    style_header(ws2)
    auto_width(ws2, max_w=14)
    ws2.freeze_panes = "C2"
    fill_reno = PatternFill("solid", fgColor=C_RENO)
    fill_free = PatternFill("solid", fgColor=C_FREE)
    for row in ws2.iter_rows(min_row=2):
        for cell in row[2:]:
            if cell.value == "EN TRAVAUX":
                cell.fill      = fill_reno
                cell.font      = Font(bold=True, size=8)
                cell.alignment = Alignment(horizontal="center")
            else:
                cell.fill = fill_free
            cell.border = border

    # Feuille 3 : Stats annuelles
    df_stats.to_excel(writer, sheet_name="3_Stats_Annuelles", index=False)
    ws3 = writer.sheets["3_Stats_Annuelles"]
    style_header(ws3)
    auto_width(ws3)
    ws3.freeze_panes = "A2"
    for i, row_ws in enumerate(ws3.iter_rows(min_row=2, max_row=len(df_stats)+1)):
        yr = df_stats.iloc[i]["annee"]
        ok_budget = df_stats.iloc[i]["budget_restant"] >= 0
        red = df_stats.iloc[i]["reduction_pct"]
        for cell in row_ws:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center")
            hdr = ws3.cell(1, cell.column).value
            if hdr == "budget_restant":
                cell.fill = PatternFill("solid", fgColor="D9EAD3" if ok_budget else "FFE0E0")
                cell.font = Font(bold=True, color="006100" if ok_budget else "9C0006")
            elif hdr == "reduction_pct" and yr in cibles:
                cell.fill = PatternFill("solid", fgColor=C_TARGET_OK if red >= cibles[yr] else C_TARGET_NOK)
                cell.font = Font(bold=True)
            elif cell.fill.fgColor.rgb in ("00000000","FFFFFFFF","00FFFFFF"):
                cell.fill = PatternFill("solid", fgColor=C_STRIPE1 if i%2==0 else C_STRIPE2)
    # Dégradé sur gain cumulé
    col_map = {c.value: get_column_letter(c.column) for c in ws3[1]}
    if "gain_cumule_mwh" in col_map:
        cl = col_map["gain_cumule_mwh"]
        ws3.conditional_formatting.add(
            f"{cl}2:{cl}{len(df_stats)+1}",
            ColorScaleRule(start_type="min", start_color="FFFFFF",
                           end_type="max",   end_color="2E75B6")
        )

    # Feuille 4 : Disponibilité par catégorie
    df_avail.to_excel(writer, sheet_name="4_Disponibilite_Cat", index=False)
    ws4 = writer.sheets["4_Disponibilite_Cat"]
    style_header(ws4)
    auto_width(ws4)
    ws4.freeze_panes = "A2"
    for i, row_ws in enumerate(ws4.iter_rows(min_row=2, max_row=len(df_avail)+1)):
        taux = df_avail.iloc[i]["taux_utilisation"]
        for cell in row_ws:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center")
            hdr = ws4.cell(1, cell.column).value
            if hdr == "taux_utilisation":
                if taux > 0.20:
                    cell.fill = PatternFill("solid", fgColor="FF0000")
                    cell.font = Font(bold=True, color="FFFFFF")
                elif taux > 0.15:
                    cell.fill = PatternFill("solid", fgColor="FFC000")
                    cell.font = Font(bold=True)
                else:
                    cell.fill = PatternFill("solid", fgColor="E2EFDA")
            elif cell.fill.fgColor.rgb in ("00000000","FFFFFFFF","00FFFFFF"):
                cell.fill = PatternFill("solid", fgColor=C_STRIPE1 if i%2==0 else C_STRIPE2)

print(f"\n✅ '{OUTPUT_FILE}' généré.")
print("  0_Résumé_Objectifs | 1_Transitions | 2_Gantt_Annuel | 3_Stats_Annuelles | 4_Disponibilite_Cat")


✅ 'plan_renovation_solution.xlsx' généré.
  0_Résumé_Objectifs | 1_Transitions | 2_Gantt_Annuel | 3_Stats_Annuelles | 4_Disponibilite_Cat


In [26]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

# ============================================================
# GANTT MENSUEL — graphe temporel
# Chaque cellule indique le statut du bâtiment ce mois :
#   ▶ Nm  = démarrage (N = durée, commentaire = travaux)
#   ░     = en cours
#   ✓     = fin de chantier (dernier mois)
#   vide  = libre
# ============================================================

OUTPUT_GANTT = "gantt_mensuel.xlsx"

MOIS_FR = ["Jan","Fév","Mar","Avr","Mai","Jun",
           "Jul","Aoû","Sep","Oct","Nov","Déc"]

# ── Couleurs ────────────────────────────────────────────────
C_HEADER_BG = "1F3864"
C_HEADER_FG = "FFFFFF"
C_START     = "C55A11"   # orange foncé : démarrage
C_ONGOING   = "FFD966"   # jaune        : en cours
C_END       = "70AD47"   # vert         : fin
C_FREE      = "F2F2F2"   # gris pâle    : libre
C_TYPE_BG   = "D6E4F0"   # bleu pâle    : colonne type

thin   = Side(style="thin",   color="CCCCCC")
thick  = Side(style="medium", color="555555")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# ── Construction de la grille : statut par (bâtiment, mois) ─
# On parcourt les arcs de rénovation actifs (f[b,a].X > 0.5)

grid = {b: [""] * T for b in building_names}   # T mois, index 0..T-1

for b in building_names:
    for a, arc in enumerate(temporal_arcs[b]):
        if arc["type"] != "renov":
            continue
        if f[b, a].X < 0.5:
            continue

        t_start = arc["from"][1]
        t_end   = arc["to"][1]        # mois de fin (exclu du chantier)
        r_from  = arc["from"][0]
        r_to    = arc["to"][0]
        duree   = t_end - t_start

        # Travaux réalisés (pour le commentaire)
        added = works_added(r_from, r_to)
        travaux_str = ", ".join(str(w) for w in added) if added else r_to

        # Mois de démarrage
        if t_start < T:
            grid[b][t_start] = f"START:{travaux_str}|{duree}m"

        # Mois intermédiaires
        for t in range(t_start + 1, min(t_end - 1, T)):
            grid[b][t] = "ONGOING"

        # Dernier mois du chantier (t_end - 1), si différent du démarrage
        t_last = t_end - 1
        if t_last < T and t_last > t_start:
            grid[b][t_last] = "END"
        elif t_last < T and t_last == t_start:
            # Chantier d'un seul mois : démarrage = fin
            grid[b][t_start] = f"START+END:{travaux_str}|{duree}m"

# ── Création du classeur ─────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Gantt_Mensuel"

# Ligne 1 : années (fusionnées sur 12 colonnes chacune)
ws.cell(1, 1, "Bâtiment").font = Font(bold=True, color=C_HEADER_FG, size=9)
ws.cell(1, 1).fill      = PatternFill("solid", fgColor=C_HEADER_BG)
ws.cell(1, 1).alignment = Alignment(horizontal="center", vertical="center")
ws.cell(1, 2, "Type").font = Font(bold=True, color=C_HEADER_FG, size=9)
ws.cell(1, 2).fill      = PatternFill("solid", fgColor=C_HEADER_BG)
ws.cell(1, 2).alignment = Alignment(horizontal="center", vertical="center")

for a_idx, yr in enumerate(range(2026, 2026 + nbr_annees)):
    col_s = 3 + a_idx * 12
    col_e = col_s + 11
    ws.merge_cells(start_row=1, start_column=col_s,
                   end_row=1,   end_column=col_e)
    cell = ws.cell(1, col_s, str(yr))
    cell.font      = Font(bold=True, color=C_HEADER_FG, size=10)
    cell.fill      = PatternFill("solid", fgColor=C_HEADER_BG)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border    = Border(left=thick, right=thick, top=thick, bottom=thick)

# Ligne 2 : mois
ws.cell(2, 1, "Bâtiment")
ws.cell(2, 2, "Type")
for col_offset in range(T):
    yr_idx  = col_offset // 12
    mois_idx = col_offset % 12
    label   = MOIS_FR[mois_idx]
    cell    = ws.cell(2, 3 + col_offset, label)
    cell.font      = Font(bold=True, color=C_HEADER_FG, size=7)
    cell.fill      = PatternFill("solid", fgColor=C_HEADER_BG)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border    = border

# Séparateur années : bordure épaisse à gauche de chaque janvier
for a_idx in range(nbr_annees):
    col = 3 + a_idx * 12
    for row in range(1, len(building_names) + 4):
        c = ws.cell(row, col)
        c.border = Border(
            left   = thick,
            right  = c.border.right,
            top    = c.border.top,
            bottom = c.border.bottom
        )

ws.row_dimensions[1].height = 16
ws.row_dimensions[2].height = 18
ws.freeze_panes = "C3"

ws.column_dimensions["A"].width = 34
ws.column_dimensions["B"].width = 14
for col in range(3, 3 + T):
    ws.column_dimensions[get_column_letter(col)].width = 4.2

# ── Remplissage des données ──────────────────────────────────
fill_start   = PatternFill("solid", fgColor=C_START)
fill_ongoing = PatternFill("solid", fgColor=C_ONGOING)
fill_end     = PatternFill("solid", fgColor=C_END)
fill_free    = PatternFill("solid", fgColor=C_FREE)
fill_type    = PatternFill("solid", fgColor=C_TYPE_BG)

for row_idx, b in enumerate(building_names, start=3):

    # Colonne bâtiment
    c_bat = ws.cell(row_idx, 1, b)
    c_bat.font      = Font(size=8, bold=True)
    c_bat.alignment = Alignment(vertical="center")
    c_bat.border    = border

    # Colonne type
    c_type = ws.cell(row_idx, 2, building_type_map[b])
    c_type.font      = Font(size=8)
    c_type.fill      = fill_type
    c_type.alignment = Alignment(horizontal="center", vertical="center")
    c_type.border    = border

    for t, status in enumerate(grid[b]):
        cell = ws.cell(row_idx, 3 + t)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border

        if status.startswith("START+END:"):
            payload     = status[10:]
            parts       = payload.split("|")
            duree_s     = parts[1] if len(parts) > 1 else ""
            travaux_s   = parts[0]
            cell.value  = f"▶✓{duree_s}"
            cell.fill   = fill_end          # vert = terminé immédiatement
            cell.font   = Font(bold=True, color="FFFFFF", size=6)
            cell.comment = Comment(
                f"Travaux : {travaux_s}\nDurée : {duree_s}",
                "Optimiseur", width=220, height=60
            )

        elif status.startswith("START:"):
            payload     = status[6:]
            parts       = payload.split("|")
            travaux_s   = parts[0]
            duree_s     = parts[1] if len(parts) > 1 else ""
            cell.value  = f"▶{duree_s}"
            cell.fill   = fill_start
            cell.font   = Font(bold=True, color="FFFFFF", size=6)
            cell.comment = Comment(
                f"Travaux : {travaux_s}\nDurée : {duree_s}",
                "Optimiseur", width=220, height=60
            )

        elif status == "ONGOING":
            cell.value = "░"
            cell.fill  = fill_ongoing
            cell.font  = Font(color="B8860B", size=7)

        elif status == "END":
            cell.value = "✓"
            cell.fill  = fill_end
            cell.font  = Font(bold=True, color="FFFFFF", size=7)

        else:
            cell.fill = fill_free

    ws.row_dimensions[row_idx].height = 13

wb.save(OUTPUT_GANTT)

print(f"✅ Gantt mensuel exporté : '{OUTPUT_GANTT}'")
print(f"   {len(building_names)} bâtiments × {T} mois")
print()
print("Légende :")
print("  ▶ Nm  = démarrage (durée N mois) — survoler pour détail travaux")
print("  ░     = en cours de rénovation")
print("  ✓     = dernier mois du chantier")
print("  ▶✓    = chantier d'un seul mois")
print("  vide  = libre")

✅ Gantt mensuel exporté : 'gantt_mensuel.xlsx'
   71 bâtiments × 300 mois

Légende :
  ▶ Nm  = démarrage (durée N mois) — survoler pour détail travaux
  ░     = en cours de rénovation
  ✓     = dernier mois du chantier
  ▶✓    = chantier d'un seul mois
  vide  = libre
